# W2D2 — 엣지 검출 비교: Sobel vs Canny vs Laplacian

## 📌 오늘의 목표
3가지 엣지 검출 함수의 **특성 차이**를 직접 눈으로 확인하고,  
어떤 결함 유형에 어떤 엣지 검출기를 선택해야 하는지 판단 기준을 세운다.

## 🎯 배울 함수
| 함수 | 핵심 파라미터 | 특징 |
|------|-------------|------|
| `cv2.Sobel` | `dx`, `dy`, `ksize` | 방향별 밝기 변화량(기울기) 계산 |
| `cv2.Canny` | `threshold1`, `threshold2` | 두 임계값으로 강한 엣지만 선택 |
| `cv2.Laplacian` | `ksize` | 모든 방향 이중 미분, 노이즈에 민감 |

## 📖 원문 자료
> 📖 원리 이해: [LearnOpenCV - Edge Detection](https://learnopencv.com/edge-detection-using-opencv/)  
> 📋 파라미터 확인: [OpenCV 공식 - Canny Edge Detector](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html)

## 🏭 실무 연결
AOI/SPI 검사에서 blur로 노이즈를 제거한 뒤 **엣지 검출**이 다음 단계다.  
결함은 주변과 밝기 차이가 나는 경계(엣지)로 나타나므로,  
엣지를 얼마나 정확하게 찾느냐가 **결함 검출률을 결정**한다.

---
## 📦 1. Import & 데이터 로딩

**왜 blur를 먼저 적용하나?**  
→ 엣지 검출은 밝기 변화에 민감하다. 노이즈도 밝기 변화이므로 blur 없이 엣지 검출하면 **노이즈가 전부 엣지로 잡힌다**.  
→ GaussianBlur로 노이즈를 먼저 제거한 뒤 엣지를 검출하는 것이 표준 파이프라인이다.

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_ROOT = Path('../data/sample_images')

img_building = cv2.imread(str(DATA_ROOT / 'building.jpg'), cv2.IMREAD_GRAYSCALE)

img_building_blur = cv2.GaussianBlur(img_building, (5, 5), 0)

print(f'이미지 크기: {img_building.shape}')
print(f'픽셀 값 범위: {img_building.min()} ~ {img_building.max()}')

---
## 🔧 2. 엣지 검출 3종 비교

### cv2.Sobel
- **원리**: 픽셀 밝기의 **기울기(gradient)** 를 계산 — 밝기가 급격히 변하는 곳이 엣지
- **파라미터**: `dx=1, dy=0` → 가로 방향 엣지 / `dx=0, dy=1` → 세로 방향 엣지
- **실무 용도**: 방향성 있는 결함(긁힘, 선형 균열) 검출

### cv2.Canny
- **원리**: Sobel로 기울기 계산 → 두 임계값(threshold)으로 강한 엣지만 선택
- **파라미터**: `threshold1` — 약한 엣지 하한선, `threshold2` — 강한 엣지 기준
- **실무 용도**: 가장 범용적인 엣지 검출, 깔끔한 단일 픽셀 선

### cv2.Laplacian
- **원리**: 모든 방향의 밝기 변화를 한 번에 계산 (이중 미분)
- **파라미터**: `ksize` — 커널 크기
- **실무 용도**: 방향 무관한 엣지 검출, 단 노이즈에 매우 민감

In [ ]:
img = img_building_blur

sobel_x = cv2.Sobel(img, cv2.CV_64F, dx=1, dy=0, ksize=3)
sobel_y = cv2.Sobel(img, cv2.CV_64F, dx=0, dy=1, ksize=3)
sobel   = cv2.magnitude(sobel_x, sobel_y)
sobel   = np.uint8(np.clip(sobel, 0, 255))

canny = cv2.Canny(img, threshold1=50, threshold2=150)

laplacian = cv2.Laplacian(img, cv2.CV_64F, ksize=3)
laplacian = np.uint8(np.clip(np.abs(laplacian), 0, 255))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
titles = ['원본 (blur 적용)', 'Sobel (X+Y)', 'Canny', 'Laplacian']
images = [img_building, sobel, canny, laplacian]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('Building — 엣지 검출 3종 비교', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
1. **균열 선** — 세 방법 모두 균열의 경계를 밝은 선으로 잡았는지
2. **배경** — 노이즈가 엣지로 잡혀서 배경이 지저분한지
3. **Canny** — 다른 두 방법보다 선이 얇고 깔끔한지

> 🤔 **판단 질문 1:** Canny가 Sobel보다 선이 얇게 나오는 이유는 뭘까요? (힌트: threshold의 역할)  
> 🤔 **판단 질문 2:** Laplacian 결과가 Sobel보다 노이즈가 많이 보인다면 왜 그럴까요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 1 [예측형]: blur 없이 원본 이미지에 바로 Canny를 적용하면 어떻게 될까? 예상 후 아래 셀에서 실행.
- 챌린지 2 [판단형]: Sobel X만 쓰면 어떤 방향의 결함이 안 잡힐까?
- 챌린지 3 [지시형]: "Canny 대신 Scratches 이미지로 같은 비교를 해줘"라고 Claude에게 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 1: blur 없이 엣지 검출
> `___` 부분을 채워서 실행하세요. 예측과 결과가 같나요?

In [ ]:
# blur 없이 원본에 바로 Canny 적용 (threshold는 blur 버전과 동일하게 설정해서 공정 비교)
canny_no_blur = cv2.Canny(img_building, threshold1=50, threshold2=150)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img_building,  cmap='gray'); axes[0].set_title('원본');              axes[0].axis('off')
axes[1].imshow(canny,         cmap='gray'); axes[1].set_title('Canny (blur 후)');   axes[1].axis('off')
axes[2].imshow(canny_no_blur, cmap='gray'); axes[2].set_title('Canny (blur 없음)'); axes[2].axis('off')
plt.suptitle('챌린지 1 — blur 전처리 유무 비교 (threshold 동일: 50/150)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 엣지 픽셀 수로 정량 비교
print(f'Canny (blur 후)   — 엣지 픽셀 수: {canny.sum() // 255:,}')
print(f'Canny (blur 없음) — 엣지 픽셀 수: {canny_no_blur.sum() // 255:,}')

In [ ]:
# 챌린지 3 — Scratches 이미지로 Sobel vs Canny vs Laplacian 비교
DEFECT_ROOT = Path('../data/kaggle/archive/NEU-DET/train/images')

img_scratches = cv2.imread(str(DEFECT_ROOT / 'scratches/scratches_1.jpg'), cv2.IMREAD_GRAYSCALE)
img_scratches_blur = cv2.GaussianBlur(img_scratches, (5, 5), 0)

sx = cv2.Sobel(img_scratches_blur, cv2.CV_64F, 1, 0, ksize=3)
sy = cv2.Sobel(img_scratches_blur, cv2.CV_64F, 0, 1, ksize=3)
scratches_sobel     = np.uint8(np.clip(cv2.magnitude(sx, sy), 0, 255))
scratches_canny     = cv2.Canny(img_scratches_blur, threshold1=50, threshold2=150)
scratches_laplacian = np.uint8(np.clip(np.abs(cv2.Laplacian(img_scratches_blur, cv2.CV_64F, ksize=3)), 0, 255))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
titles = ['Scratches 원본', 'Sobel (X+Y)', 'Canny', 'Laplacian']
images = [img_scratches, scratches_sobel, scratches_canny, scratches_laplacian]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('챌린지 3 — Scratches 엣지 검출 3종 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 3. Sobel 방향별 비교 — X, Y, 합산

**왜 방향을 나눠서 보나?**  
→ 결함의 방향에 따라 잡히는 엣지가 다르다.  
→ 세로 방향 결함은 Sobel X에서 강하게 나타나고, 가로 방향 결함은 Sobel Y에서 강하게 나타난다.  
→ 두 방향을 합산하면 방향에 무관하게 모든 엣지를 잡을 수 있다.

In [ ]:
img = img_building_blur

sobel_x_raw = cv2.Sobel(img, cv2.CV_64F, dx=1, dy=0, ksize=3)
sobel_y_raw = cv2.Sobel(img, cv2.CV_64F, dx=0, dy=1, ksize=3)
sobel_xy    = cv2.magnitude(sobel_x_raw, sobel_y_raw)

sobel_x_show  = np.uint8(np.clip(np.abs(sobel_x_raw), 0, 255))
sobel_y_show  = np.uint8(np.clip(np.abs(sobel_y_raw), 0, 255))
sobel_xy_show = np.uint8(np.clip(sobel_xy,             0, 255))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
titles = ['원본', 'Sobel X (세로 엣지)', 'Sobel Y (가로 엣지)', 'Sobel X+Y (합산)']
images = [img_building, sobel_x_show, sobel_y_show, sobel_xy_show]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Sobel 방향별 비교 — Building', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **Sobel X** — 세로로 뻗은 균열 선이 잘 잡히는지
- **Sobel Y** — 가로로 뻗은 균열 선이 잡히는지
- **합산** — X, Y 중 하나만 쓸 때보다 더 많은 엣지가 잡히는지

> 🤔 **판단 질문 3:** Crazing 균열이 특정 방향으로 치우쳐 있다면, X와 Y 중 어느 쪽이 더 밝게 나올까요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 4 [예측형]: ksize를 3에서 5로 키우면 Sobel 결과가 어떻게 변할까? 예상 후 실행.
- 챌린지 5 [판단형]: Sobel X+Y 합산이 Canny보다 선이 두꺼운 이유는?
```

#### ✏️ 빈칸 채우기 — 챌린지 4: Sobel ksize 변화

In [ ]:
# ksize=5로 변경
sobel_x5 = cv2.Sobel(img, cv2.CV_64F, dx=___, dy=___, ksize=___)
sobel_y5 = cv2.Sobel(img, cv2.CV_64F, dx=___, dy=___, ksize=___)
sobel_5  = cv2.magnitude(sobel_x5, sobel_y5)
sobel_5  = np.uint8(np.clip(sobel_5, 0, 255))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sobel_xy_show, cmap='gray'); axes[0].set_title('Sobel (ksize=3)'); axes[0].axis('off')
axes[1].imshow(sobel_5,       cmap='gray'); axes[1].set_title('Sobel (ksize=5)'); axes[1].axis('off')
plt.suptitle('Sobel ksize 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 4. Canny threshold 파라미터 실험

**threshold가 하는 일:**  
→ Canny는 내부적으로 Sobel로 기울기를 계산한 뒤, 두 임계값으로 엣지를 선별한다.  
→ `threshold2`(상한) 이상 → **강한 엣지** — 무조건 포함  
→ `threshold1`(하한) ~ `threshold2` 사이 → **약한 엣지** — 강한 엣지에 연결된 경우만 포함  
→ `threshold1` 미만 → **버림**

**실무 팁:** threshold2 : threshold1 = 3:1 또는 2:1 비율이 일반적 출발점이다.

In [ ]:
img = img_building_blur

params = [(30, 90), (50, 150), (100, 200), (150, 250)]

fig, axes = plt.subplots(1, len(params) + 1, figsize=(24, 5))
axes[0].imshow(img_building, cmap='gray')
axes[0].set_title('원본', fontsize=12)
axes[0].axis('off')

for ax, (t1, t2) in zip(axes[1:], params):
    edge = cv2.Canny(img, threshold1=t1, threshold2=t2)
    ax.imshow(edge, cmap='gray')
    ax.set_title(f'Canny\nt1={t1}, t2={t2}', fontsize=11)
    ax.axis('off')

plt.suptitle('Canny — threshold 파라미터 변화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- threshold가 낮을 때 → 배경 노이즈가 엣지로 잡히는지
- threshold가 높을 때 → 결함 선이 끊겨서 사라지는지
- 균열 선이 유지되면서 배경은 깨끗한 **최적 threshold** 범위는?

> 🤔 **판단 질문 4:** threshold1을 너무 낮게 설정하면 어떤 문제가 생기고, 너무 높게 설정하면 어떤 문제가 생기나요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 6 [예측형]: threshold1=200, threshold2=50 처럼 순서를 바꾸면(t1 > t2) 어떻게 될까?
- 챌린지 7 [판단형]: 제조 검사 라인에서 threshold를 고정값으로 쓰면 왜 위험할까? (힌트: 조명 변화)
```

#### ✏️ 빈칸 채우기 — 챌린지 6: threshold 순서 역전

In [ ]:
# threshold1 > threshold2 로 뒤집어서 실행
canny_reversed = cv2.Canny(img, threshold1=___, threshold2=___)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(canny,          cmap='gray'); axes[0].set_title('정상 (t1=50, t2=150)');   axes[0].axis('off')
axes[1].imshow(canny_reversed, cmap='gray'); axes[1].set_title('역전 (t1=150, t2=50)'); axes[1].axis('off')
plt.suptitle('Canny threshold 역전 실험', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 5. 영역별 엣지 검출 비교

**왜 영역별로 비교하나?**  
→ 건물 이미지는 수직 엣지(벽, 기둥)가 많은 영역과 수평 엣지(창문 가로선, 지붕선)가 많은 영역이 섞여 있다.  
→ 이미지를 상/하로 나누면 Sobel X와 Y 중 어느 쪽이 강하게 반응하는지 방향별 특성을 직접 확인할 수 있다.

In [ ]:
h, w = img_building.shape
region_top    = img_building[:h//2, :]   # 상단 절반
region_bottom = img_building[h//2:, :]   # 하단 절반

regions = {
    '상단 (Top)':    (region_top,    cv2.GaussianBlur(region_top,    (5, 5), 0)),
    '하단 (Bottom)': (region_bottom, cv2.GaussianBlur(region_bottom, (5, 5), 0)),
}

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
col_titles = ['원본', 'Sobel (X+Y)', 'Canny', 'Laplacian']

for row_idx, (region_name, (img_src, img_b)) in enumerate(regions.items()):
    sx = cv2.Sobel(img_b, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(img_b, cv2.CV_64F, 0, 1, ksize=3)
    s  = np.uint8(np.clip(cv2.magnitude(sx, sy), 0, 255))
    c  = cv2.Canny(img_b, 50, 150)
    l  = np.uint8(np.clip(np.abs(cv2.Laplacian(img_b, cv2.CV_64F, ksize=3)), 0, 255))

    for col_idx, image in enumerate([img_src, s, c, l]):
        axes[row_idx][col_idx].imshow(image, cmap='gray')
        axes[row_idx][col_idx].set_title(f'{region_name}\n{col_titles[col_idx]}', fontsize=11)
        axes[row_idx][col_idx].axis('off')

plt.suptitle('Building — 영역별 엣지 검출 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **Scratches 행**: Sobel이 긁힌 선의 방향을 잘 잡는지, Canny는 선이 얼마나 깔끔한지
- **Pitted Surface 행**: 작은 구덩이의 경계가 세 방법 중 어디서 가장 뚜렷하게 잡히는지

> 🤔 **판단 질문 5:** Scratches에 Sobel X만 쓴다면 가로 방향 긁힘은 어떻게 될까요? 어떻게 해결하면 좋을까요?  
> 🤔 **판단 질문 6:** Pitted Surface의 작은 구덩이가 Laplacian에서 더 잘 잡힌다면, 왜 그럴까요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 8 [판단형]: Scratches에 medianBlur(W2D1)를 전처리로 쓰고 Canny를 적용하면 GaussianBlur를 쓸 때와 결과가 달라질까?
- 챌린지 9 [지시형]: "crazing도 추가해서 3종 결함 전체 비교 그리드를 만들어줘"라고 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 8: 전처리 blur 교체 실험
> Scratches에 GaussianBlur 대신 medianBlur를 전처리로 쓰고 Canny를 적용하세요.

In [ ]:
# 전처리 blur 교체 실험 — ___ 부분을 채우세요
pre_gaussian = cv2.GaussianBlur(img_building, (5, 5), 0)
pre_median   = cv2.___(img_building, ___)

canny_gauss  = cv2.Canny(pre_gaussian, 50, 150)
canny_median = cv2.Canny(___, 50, 150)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(canny_gauss,  cmap='gray'); axes[0].set_title('GaussianBlur → Canny'); axes[0].axis('off')
axes[1].imshow(canny_median, cmap='gray'); axes[1].set_title('medianBlur → Canny');   axes[1].axis('off')
plt.suptitle('Building — 전처리 blur 교체 실험', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📝 오늘 배운 것 정리

### 핵심 의사결정 기준
| 상황 | 추천 엣지 검출기 | 이유 |
|------|----------------|------|
| 방향성 있는 결함 (긁힘, 선형 균열) | Sobel X 또는 Y | 특정 방향 엣지에 민감 |
| 범용 결함 검출, 깔끔한 경계선 | Canny | 두 threshold로 노이즈 엣지 제거 |
| 방향 무관, 점형 결함 경계 | Laplacian | 모든 방향 이중 미분, 단 노이즈 민감 |

### 표준 파이프라인
```
원본 이미지
    → GaussianBlur (노이즈 제거)
    → 엣지 검출 (Sobel / Canny / Laplacian)
    → 이후 단계 (이진화, 윤곽선 검출 등)
```

### 처리 속도 비교 (참고)
- Sobel < Canny < Laplacian 순으로 복잡도 증가
- 실시간 검사 라인에서는 Sobel이 속도 면에서 유리

### 복습 퀴즈
1. Canny의 threshold1과 threshold2 역할은 각각 무엇인가?
2. Sobel X만 쓰면 어떤 방향의 결함을 놓치게 되는가?
3. 엣지 검출 전에 blur를 먼저 적용하는 이유는?